# 06 - Snowpark: Python dentro do Snowflake

**Objetivo:** fazer transformações com **Python** em vez de SQL e ver que o resultado é o mesmo.

**Analogia:** SQL e Python são dois idiomas diferentes para dar a mesma ordem ao mesmo cozinheiro. Com o Snowpark, você escreve em Python, mas quem cozinha continua sendo o Snowflake: os dados **não saem** de lá.

```mermaid
flowchart LR
  PY[Seu codigo Python] -->|traduzido para SQL| SF[Motor Snowflake]
  SF -->|resultado| PY
```

**Tempo estimado:** 20 minutos

In [ ]:
# Pega a sessao do Snowflake que o notebook ja abriu para voce
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()
session.use_warehouse("ESPM_WH")
session.use_database("ESPM_HOL")
session.use_schema("GOLD")
print("Conectado como", session.get_current_role(), "em", session.get_current_database())

## Passo 1 - DataFrame: uma tabela vista pelo Python

Um **DataFrame** é uma tabela que o Python consegue manipular. Ele é *preguiçoso*: só executa quando você pede para ver (`.show()`) ou salvar.

In [ ]:
rfm = session.table("ESPM_HOL.GOLD.RFM_CLIENTES")
rfm.show(5)
print("Total de clientes com compra valida:", rfm.count())   # esperado: 191

### EXERCICIO 1 - Filtre só os clientes do segmento CAMPEOES

Complete os `???` na célula abaixo.

**Dica:** os segmentos possíveis são CAMPEOES, FIEIS, NOVOS e EM_RISCO.

**Travou?** Abra o Cortex Code e peça:
> Em Snowpark Python, como filtro o DataFrame rfm para SEGMENTO igual a CAMPEOES?

In [ ]:
campeoes = rfm.filter(F.col("SEGMENTO") == "???")
campeoes.select("CLIENTE_ID", "NOME", "FREQUENCIA", "VALOR_TOTAL").show()

### EXERCICIO 2 - Conte quantos clientes existem em cada segmento

Complete os `???` na célula abaixo.

**Dica:** é o equivalente ao GROUP BY do SQL: agrupe pela coluna de segmento.

**Travou?** Abra o Cortex Code e peça:
> Em Snowpark, agrupe o DataFrame rfm por SEGMENTO e conte os clientes

In [ ]:
por_segmento = rfm.group_by("???").agg(F.count(F.lit(1)).alias("CLIENTES"))
por_segmento.sort(F.col("CLIENTES").desc()).show()

## Passo 2 - UDF: criando sua própria função

Uma **UDF** (*User Defined Function*) é uma função que você escreve em Python e depois usa **até dentro de SQL**, como se fosse nativa.

Regra de negócio do CRM: classificar o cliente pelo valor total gasto.

| Valor gasto | Faixa |
|---|---|
| até R$ 999,99 | BASICO |
| R$ 1.000 a R$ 2.999,99 | INTERMEDIARIO |
| R$ 3.000 ou mais | PREMIUM |

In [ ]:
from decimal import Decimal
from snowflake.snowpark.types import StringType, DecimalType

# Pasta (stage) onde o Snowflake guarda o codigo da funcao
session.sql("CREATE STAGE IF NOT EXISTS ESPM_HOL.PUBLIC.UDF_STAGE").collect()

def faixa_valor(valor: Decimal) -> str:
    if valor is None:
        return "SEM_COMPRA"
    if valor < Decimal("1000"):
        return "BASICO"
    if valor < Decimal("3000"):
        return "INTERMEDIARIO"
    return "PREMIUM"

faixa_udf = session.udf.register(
    func=faixa_valor,
    name="ESPM_HOL.PUBLIC.FAIXA_VALOR",
    return_type=StringType(),
    input_types=[DecimalType(12, 2)],
    is_permanent=True,                       # fica salva para outras pessoas usarem
    stage_location="@ESPM_HOL.PUBLIC.UDF_STAGE",
    replace=True,
)
print("UDF criada!")

A mesma função, chamada **em SQL**:

In [ ]:
%%sql -r dataframe_6
SELECT ESPM_HOL.PUBLIC.FAIXA_VALOR(500)  AS faixa_500,
       ESPM_HOL.PUBLIC.FAIXA_VALOR(1500) AS faixa_1500,
       ESPM_HOL.PUBLIC.FAIXA_VALOR(5000) AS faixa_5000;

## Passo 3 - Salvar o resultado como tabela Gold

In [ ]:
segmentos = rfm.with_column("FAIXA_VALOR", faixa_udf(F.col("VALOR_TOTAL")))
segmentos.write.save_as_table("ESPM_HOL.GOLD.SEGMENTOS_CLIENTES", mode="overwrite")

# Checkpoint: esperado 191 linhas
resumo = session.table("ESPM_HOL.GOLD.SEGMENTOS_CLIENTES") \
    .group_by("SEGMENTO", "FAIXA_VALOR").agg(F.count(F.lit(1)).alias("CLIENTES")) \
    .sort("SEGMENTO", "FAIXA_VALOR")
resumo.show()
print("Linhas:", session.table("ESPM_HOL.GOLD.SEGMENTOS_CLIENTES").count())

## Resumo

- **Snowpark** = Python que roda **dentro** do Snowflake (pushdown).
- **UDF** = sua lógica de negócio reutilizável em Python **e** em SQL.

**Próximo:** `07_git_dbt` - versionar transformações como engenharia de software.